In [5]:
# Paso 1: importar las librerias necesarias para explorar los datos
 
import pandas as pd
import numpy as np
from pathlib import Path
 
# Ignorar warnings
# ==============================================================================
import warnings
warnings.filterwarnings("ignore")


# Configuración
# -----------------------------------------------------------------------
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [6]:
df = pd.read_csv(r"C:\Users\claud\GIT\proyecto_final\data\raw\brazil_public_holidays.csv", sep=";")

df.to_csv(
    r"C:\Users\claud\GIT\proyecto_final\data\raw\brazil_public_holidays_comas.csv",
    index=False,
    sep=","
)

# Exploración inicial de los datos

Objetivo:

- Conocer la estructura de los datasets.
- Identificar columnas y tipos de datos.
- Detectar valores nulos.
- Detectar duplicados.
- Identificar posibles problemas de calidad.
- Analizar las claves para la futura unión de los datasets.

In [7]:
# Paso 2: definir las rutas de entrada de los datasets originales

RUTA_RAW = Path("../data/raw")

archivos = {
    "customers": RUTA_RAW / "olist_customers_dataset.csv",
    "orders": RUTA_RAW / "olist_orders_dataset.csv",
    "items": RUTA_RAW / "olist_order_items_dataset.csv",
    "payments": RUTA_RAW / "olist_order_payments_dataset.csv",
    "products": RUTA_RAW / "olist_products_dataset.csv",
    "categories": RUTA_RAW / "product_category_name_translation.csv",
    "holidays": RUTA_RAW / "brazil_public_holidays_comas.csv"
}

In [8]:
# Paso 3: cargar todos los datasets en un diccionario de DataFrames

def cargar_datasets(diccionario_archivos):
    """
    Carga varios archivos CSV y los devuelve en un diccionario de DataFrames.

    Parametros:
        diccionario_archivos: diccionario con el nombre del dataset y su ruta.

    Devuelve:
        diccionario con los datasets cargados como DataFrames.
    """
    datasets = {}

    for nombre, ruta in diccionario_archivos.items():
        try:
            datasets[nombre] = pd.read_csv(ruta)
            print(f"Dataset cargado correctamente: {nombre}")
        except FileNotFoundError:
            print(f"No se encontro el archivo: {ruta}")

    return datasets


datasets = cargar_datasets(archivos)

Dataset cargado correctamente: customers
Dataset cargado correctamente: orders
Dataset cargado correctamente: items
Dataset cargado correctamente: payments
Dataset cargado correctamente: products
Dataset cargado correctamente: categories
Dataset cargado correctamente: holidays


In [9]:
# Paso 4: asignar los DataFrames a variables individuales para trabajar mejor

customers = datasets["customers"]
orders = datasets["orders"]
items = datasets["items"]
payments = datasets["payments"]
products = datasets["products"]
categories = datasets["categories"]
holidays = datasets["holidays"]

In [10]:
# Paso 5: mostrar el numero de filas y columnas de cada dataset

def mostrar_dimensiones(datasets):
    """
    Muestra el numero de filas y columnas de cada DataFrame.

    Parametros:
        datasets: diccionario de DataFrames.

    Devuelve:
        DataFrame resumen con filas y columnas.
    """
    resumen = []

    for nombre, df in datasets.items():
        resumen.append({
            "dataset": nombre,
            "filas": df.shape[0],
            "columnas": df.shape[1]
        })

    return pd.DataFrame(resumen).sort_values(by="filas", ascending=False)


resumen_dimensiones = mostrar_dimensiones(datasets)
resumen_dimensiones

,dataset,filas,columnas
2,items,112650,7
3,payments,103886,5
0,customers,99441,5
1,orders,99441,8
4,products,32951,9
6,holidays,1670,7
5,categories,71,2


In [11]:
# Paso 6: visualizar las primeras filas de cada dataset para entender su estructura

for nombre, df in datasets.items():
    print(f"\nDataset: {nombre}")
    display(df.head())


Dataset: customers


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



Dataset: orders


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Dataset: items


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0



Dataset: categories


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor



Dataset: holidays


,Unnamed: 0,countryOrRegion,holidayName,normalizeHolidayName,isPaidTimeOff,countryRegionCode,date
0,29,Brazil,Carnaval,Carnaval,NaN,BR,10/02/1970
1,30,Brazil,Quarta-feira de cinzas (InÃ­cio da Quaresma),Quarta-feira de cinzas (InÃ­cio da Quaresma),NaN,BR,11/02/1970
2,61,Brazil,Sexta-feira Santa,Sexta-feira Santa,NaN,BR,27/03/1970
3,81,Brazil,PÃ¡scoa,PÃ¡scoa,NaN,BR,29/03/1970
4,125,Brazil,Tiradentes,Tiradentes,NaN,BR,21/04/1970


In [12]:
# Paso 7: mostrar las columnas disponibles en cada dataset

def mostrar_columnas(datasets):
    """
    Muestra las columnas de cada dataset.

    Parametros:
        datasets: diccionario de DataFrames.
    """
    for nombre, df in datasets.items():
        print(f"\nColumnas de {nombre}:")
        print(df.columns.tolist())


mostrar_columnas(datasets)


Columnas de customers:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Columnas de orders:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Columnas de items:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Columnas de payments:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Columnas de products:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Columnas de categories:
['product_category_name', 'product_category_name_english']

Columnas de holidays:
['Unnamed: 0', 'countryOrRegion', 'holidayName', 'normalizeHolidayName', 'isPaidTimeOff', 'coun

In [13]:
# Paso 8: analizar los tipos de datos de cada columna

def resumen_tipos_datos(datasets):
    """
    Genera un resumen de los tipos de datos por dataset.

    Parametros:
        datasets: diccionario de DataFrames.

    Devuelve:
        DataFrame con dataset, columna y tipo de dato.
    """
    resumen = []

    for nombre, df in datasets.items():
        for columna, tipo in df.dtypes.items():
            resumen.append({
                "dataset": nombre,
                "columna": columna,
                "tipo_dato": tipo
            })

    return pd.DataFrame(resumen)


tipos_datos = resumen_tipos_datos(datasets)
tipos_datos

,dataset,columna,tipo_dato
0,customers,customer_id,object
1,customers,customer_unique_id,object
2,customers,customer_zip_code_prefix,int64
3,customers,customer_city,object
4,customers,customer_state,object
5,orders,order_id,object
6,orders,customer_id,object
7,orders,order_status,object
8,orders,order_purchase_timestamp,object
9,orders,order_approved_at,object


In [14]:
# Paso 9: calcular valores nulos y porcentaje de nulos por columna

def resumen_nulos(datasets):
    """
    Calcula el numero y porcentaje de nulos por columna.

    Parametros:
        datasets: diccionario de DataFrames.

    Devuelve:
        DataFrame con resumen de nulos.
    """
    resumen = []

    for nombre, df in datasets.items():
        total_filas = len(df)

        for columna in df.columns:
            nulos = df[columna].isna().sum()
            porcentaje = (nulos / total_filas) * 100

            resumen.append({
                "dataset": nombre,
                "columna": columna,
                "nulos": nulos,
                "porcentaje_nulos": round(porcentaje, 2)
            })

    return pd.DataFrame(resumen).sort_values(
        by="porcentaje_nulos",
        ascending=False
    )


nulos = resumen_nulos(datasets)
nulos

,dataset,columna,nulos,porcentaje_nulos
40,holidays,isPaidTimeOff,1670,100.00
11,orders,order_delivered_customer_date,2965,2.98
27,products,product_name_lenght,610,1.85
26,products,product_category_name,610,1.85
28,products,product_description_lenght,610,1.85
29,products,product_photos_qty,610,1.85
10,orders,order_delivered_carrier_date,1783,1.79
9,orders,order_approved_at,160,0.16
32,products,product_height_cm,2,0.01
31,products,product_length_cm,2,0.01


In [15]:
# Paso 10: filtrar solo las columnas que tienen valores nulos

nulos_con_valores = nulos[nulos["nulos"] > 0]
nulos_con_valores

,dataset,columna,nulos,porcentaje_nulos
40,holidays,isPaidTimeOff,1670,100.00
11,orders,order_delivered_customer_date,2965,2.98
27,products,product_name_lenght,610,1.85
26,products,product_category_name,610,1.85
28,products,product_description_lenght,610,1.85
29,products,product_photos_qty,610,1.85
10,orders,order_delivered_carrier_date,1783,1.79
9,orders,order_approved_at,160,0.16
32,products,product_height_cm,2,0.01
31,products,product_length_cm,2,0.01


In [16]:
# Paso 11: comprobar si existen filas duplicadas en cada dataset

def resumen_duplicados(datasets):
    """
    Calcula el numero de filas duplicadas por dataset.

    Parametros:
        datasets: diccionario de DataFrames.

    Devuelve:
        DataFrame con el numero de duplicados.
    """
    resumen = []

    for nombre, df in datasets.items():
        duplicados = df.duplicated().sum()

        resumen.append({
            "dataset": nombre,
            "filas": len(df),
            "duplicados": duplicados
        })

    return pd.DataFrame(resumen)


duplicados = resumen_duplicados(datasets)
duplicados

,dataset,filas,duplicados
0,customers,99441,0
1,orders,99441,0
2,items,112650,0
3,payments,103886,0
4,products,32951,0
5,categories,71,0
6,holidays,1670,0


In [17]:
# Paso 12: analizar la unicidad de las claves principales

claves = {
    "customers": "customer_id",
    "orders": "order_id",
    "items": "order_id",
    "payments": "order_id",
    "products": "product_id",
    "categories": "product_category_name"
}

def analizar_claves(datasets, claves):
    """
    Analiza si las claves principales son unicas o repetidas.

    Parametros:
        datasets: diccionario de DataFrames.
        claves: diccionario con dataset y columna clave.

    Devuelve:
        DataFrame con total de filas, valores unicos y repetidos.
    """
    resumen = []

    for nombre, clave in claves.items():
        df = datasets[nombre]

        resumen.append({
            "dataset": nombre,
            "clave": clave,
            "filas": len(df),
            "valores_unicos": df[clave].nunique(),
            "valores_repetidos": len(df) - df[clave].nunique()
        })

    return pd.DataFrame(resumen)


resumen_claves = analizar_claves(datasets, claves)
resumen_claves

,dataset,clave,filas,valores_unicos,valores_repetidos
0,customers,customer_id,99441,99441,0
1,orders,order_id,99441,99441,0
2,items,order_id,112650,98666,13984
3,payments,order_id,103886,99440,4446
4,products,product_id,32951,32951,0
5,categories,product_category_name,71,71,0


In [18]:
# Paso 13: obtener estadisticos descriptivos de las columnas numericas

for nombre, df in datasets.items():
    print(f"\nEstadisticos numericos de {nombre}")
    display(df.describe())


Estadisticos numericos de customers


,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000



Estadisticos numericos de orders


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-08-02 12:05:26,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 19:36:48,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522



Estadisticos numericos de items


...

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000



Estadisticos numericos de categories


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1



Estadisticos numericos de holidays


,Unnamed: 0,isPaidTimeOff
count,1670.000000,0.0
mean,34107.552096,NaN
std,20320.145962,NaN
min,29.000000,NaN
25%,16221.250000,NaN
50%,33926.000000,NaN
75%,51753.250000,NaN
max,69472.000000,NaN


In [19]:
# Paso 14: obtener resumen de columnas categoricas

for nombre, df in datasets.items():
    print(f"\nEstadisticos categoricos de {nombre}")
    display(df.describe(include="object"))


Estadisticos categoricos de customers


,customer_id,customer_unique_id,customer_city,customer_state
count,99441,99441,99441,99441
unique,99441,96096,4119,27
top,274fa6071e5e17fe303b9748641082c8,8d50f5eadf50201ccdcedfb9e2ac8455,sao paulo,SP
freq,1,17,15540,41746



Estadisticos categoricos de orders


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-08-02 12:05:26,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 19:36:48,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522



Estadisticos categoricos de items


...

,product_id,product_category_name
count,32951,32341
unique,32951,73
top,106392145fca363410d287a815be6de4,cama_mesa_banho
freq,1,3029



Estadisticos categoricos de categories


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1



Estadisticos categoricos de holidays


,countryOrRegion,holidayName,normalizeHolidayName,countryRegionCode,date
count,1670,1670,1670,1670,1670
unique,1,15,15,1,1670
top,Brazil,Carnaval,Carnaval,BR,25/12/2098
freq,1670,129,129,1670,1


In [20]:
# Paso 15: explorar las columnas de fecha del dataset de pedidos

columnas_fecha_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[columnas_fecha_orders].head()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [21]:
# Paso 16: convertir temporalmente la fecha de compra para ver el rango temporal

orders_temp = orders.copy()

orders_temp["order_purchase_timestamp"] = pd.to_datetime(
    orders_temp["order_purchase_timestamp"],
    errors="coerce"
)

fecha_minima = orders_temp["order_purchase_timestamp"].min()
fecha_maxima = orders_temp["order_purchase_timestamp"].max()

print("Fecha minima de compra:", fecha_minima)
print("Fecha maxima de compra:", fecha_maxima)

Fecha minima de compra: 2016-09-04 21:15:19
Fecha maxima de compra: 2018-10-17 17:30:18


In [22]:
# Paso 17: analizar la distribucion de estados de los pedidos

orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [23]:
# Paso 17.1: analizar el porcentaje de cada estado de pedido

orders["order_status"].value_counts(normalize=True).mul(100).round(2)

order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64

In [24]:
# Paso 18: analizar los tipos de pago disponibles

payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [25]:
# Paso 18.1: revisar estadisticos de valor de pago

payments["payment_value"].describe()

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [26]:
# Paso 19: revisar precio, coste de envio y numero de items

items[["price", "freight_value", "order_item_id"]].describe()

,price,freight_value,order_item_id
count,112650.000000,112650.000000,112650.000000
mean,120.653739,19.990320,1.197834
std,183.633928,15.806405,0.705124
min,0.850000,0.000000,1.000000
25%,39.900000,13.080000,1.000000
50%,74.990000,16.260000,1.000000
75%,134.900000,21.150000,1.000000
max,6735.000000,409.680000,21.000000


In [27]:
# Paso 20: analizar la distribucion geografica de clientes por estado

customers["customer_state"].value_counts().head(10)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

In [28]:
# Paso 21: analizar categorias de producto mas frecuentes

products["product_category_name"].value_counts().head(10)

product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64

In [29]:
# Paso 22: revisar estructura del dataset de festivos de Brasil

holidays.head()

,Unnamed: 0,countryOrRegion,holidayName,normalizeHolidayName,isPaidTimeOff,countryRegionCode,date
0,29,Brazil,Carnaval,Carnaval,NaN,BR,10/02/1970
1,30,Brazil,Quarta-feira de cinzas (InÃ­cio da Quaresma),Quarta-feira de cinzas (InÃ­cio da Quaresma),NaN,BR,11/02/1970
2,61,Brazil,Sexta-feira Santa,Sexta-feira Santa,NaN,BR,27/03/1970
3,81,Brazil,PÃ¡scoa,PÃ¡scoa,NaN,BR,29/03/1970
4,125,Brazil,Tiradentes,Tiradentes,NaN,BR,21/04/1970


In [30]:
# Paso 22.1: revisar columnas y tipos de datos de festivos

holidays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670 entries, 0 to 1669
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            1670 non-null   int64  
 1   countryOrRegion       1670 non-null   object 
 2   holidayName           1670 non-null   object 
 3   normalizeHolidayName  1670 non-null   object 
 4   isPaidTimeOff         0 non-null      float64
 5   countryRegionCode     1670 non-null   object 
 6   date                  1670 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 91.5+ KB


In [31]:
# Paso 23: convertir la fecha de festivos y revisar rango temporal

holidays_temp = holidays.copy()

holidays_temp["date"] = pd.to_datetime(
    holidays_temp["date"],
    errors="coerce"
)

print("Fecha minima de festivo:", holidays_temp["date"].min())
print("Fecha maxima de festivo:", holidays_temp["date"].max())

Fecha minima de festivo: 1970-01-05 00:00:00
Fecha maxima de festivo: 2098-12-10 00:00:00


In [32]:
# Paso 24: calcular numero de festivos por año

holidays_temp["year"] = holidays_temp["date"].dt.year

holidays_temp["year"].value_counts().sort_index()

year
1970.0    6
1971.0    8
1972.0    7
1973.0    7
1974.0    6
         ..
2094.0    8
2095.0    7
2096.0    5
2097.0    6
2098.0    7
Name: count, Length: 129, dtype: int64

In [33]:
# Paso 25: comprobar si los años de pedidos existen en el dataset de festivos

orders_temp["year"] = orders_temp["order_purchase_timestamp"].dt.year

anios_orders = sorted(orders_temp["year"].dropna().unique())
anios_holidays = sorted(holidays_temp["year"].dropna().unique())

print("Anios en orders:")
print(anios_orders)

print("\nAnios en holidays:")
print(anios_holidays)

print("\nAnios comunes:")
print(sorted(set(anios_orders).intersection(set(anios_holidays))))

Anios en orders:
[np.int32(2016), np.int32(2017), np.int32(2018)]

Anios en holidays:
[np.float64(1970.0), np.float64(1971.0), np.float64(1972.0), np.float64(1973.0), np.float64(1974.0), np.float64(1975.0), np.float64(1976.0), np.float64(1977.0), np.float64(1978.0), np.float64(1979.0), np.float64(1980.0), np.float64(1981.0), np.float64(1982.0), np.float64(1983.0), np.float64(1984.0), np.float64(1985.0), np.float64(1986.0), np.float64(1987.0), np.float64(1988.0), np.float64(1989.0), np.float64(1990.0), np.float64(1991.0), np.float64(1992.0), np.float64(1993.0), np.float64(1994.0), np.float64(1995.0), np.float64(1996.0), np.float64(1997.0), np.float64(1998.0), np.float64(1999.0), np.float64(2000.0), np.float64(2001.0), np.float64(2002.0), np.float64(2003.0), np.float64(2004.0), np.float64(2005.0), np.float64(2006.0), np.float64(2007.0), np.float64(2008.0), np.float64(2009.0), np.float64(2010.0), np.float64(2011.0), np.float64(2012.0), np.float64(2013.0), np.float64(2014.0), np.float64(20

In [34]:
# Paso 26: comprobar las claves que permitiran unir los datasets

print("orders - customers")
print("customer_id en orders:", orders["customer_id"].nunique())
print("customer_id en customers:", customers["customer_id"].nunique())

print("\norders - items")
print("order_id en orders:", orders["order_id"].nunique())
print("order_id en items:", items["order_id"].nunique())

print("\norders - payments")
print("order_id en orders:", orders["order_id"].nunique())
print("order_id en payments:", payments["order_id"].nunique())

print("\nitems - products")
print("product_id en items:", items["product_id"].nunique())
print("product_id en products:", products["product_id"].nunique())

print("\nproducts - categories")
print("product_category_name en products:", products["product_category_name"].nunique())
print("product_category_name en categories:", categories["product_category_name"].nunique())

orders - customers
customer_id en orders: 99441
customer_id en customers: 99441

orders - items
order_id en orders: 99441
order_id en items: 98666

orders - payments
order_id en orders: 99441
order_id en payments: 99440

items - products
product_id en items: 32951
product_id en products: 32951

products - categories
product_category_name en products: 73
product_category_name en categories: 71


In [35]:
# Paso 27: crear una tabla resumen general de calidad de datos

def resumen_calidad_datos(datasets):
    """
    Crea un resumen general de calidad para cada dataset.

    Parametros:
        datasets: diccionario de DataFrames.

    Devuelve:
        DataFrame con filas, columnas, nulos y duplicados.
    """
    resumen = []

    for nombre, df in datasets.items():
        total_nulos = df.isna().sum().sum()
        total_duplicados = df.duplicated().sum()

        resumen.append({
            "dataset": nombre,
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "total_nulos": total_nulos,
            "total_duplicados": total_duplicados
        })

    return pd.DataFrame(resumen)


resumen_calidad = resumen_calidad_datos(datasets)
resumen_calidad

,dataset,filas,columnas,total_nulos,total_duplicados
0,customers,99441,5,0,0
1,orders,99441,8,4908,0
2,items,112650,7,0,0
3,payments,103886,5,0,0
4,products,32951,9,2448,0
5,categories,71,2,0,0
6,holidays,1670,7,1670,0


customers
(customer_id)
       │
       ▼
orders
(order_id)
       │
       ├──────── payments
       │           (1:N)
       │
       ▼
items
(1:N)
       │
       ▼
products
(product_id)
       │
       ▼
categories

## Conclusiones de la exploracion inicial

# 1. Los datasets presentan una estructura relacional consistente

Se ha identificado un modelo de datos propio de una plataforma de comercio electrónico compuesto por varias tablas relacionadas entre sí mediante claves únicas.

Las principales relaciones detectadas son:
 
customers → orders → items → products → categories
                  ↓
               payments
                
Las claves primarias de las tablas maestras (customer_id, order_id, product_id) son únicas y no presentan duplicados, lo que garantiza la integridad referencial del modelo.
 
# 2. La calidad general de los datos es muy alta

La exploración inicial muestra que los datos presentan un buen nivel de calidad:

No se han detectado duplicados completos en las tablas principales.
Las claves primarias son únicas.
Las relaciones entre tablas son coherentes.
El porcentaje de valores nulos es muy reducido.

Esto permitirá centrar el esfuerzo en el análisis y la generación de valor, en lugar de dedicar gran parte del tiempo a resolver problemas de calidad de datos.
 
# 3. Los valores nulos son escasos y coherentes con el negocio

Los nulos identificados no parecen corresponder a errores de captura de datos.

Dataset holidays

La columna:
isPaidTimeOff
 
presenta un 100% de valores nulos y no aporta información relevante, por lo que podrá eliminarse.
 
Dataset orders

Los nulos observados en:
order_delivered_customer_date
order_delivered_carrier_date
order_approved_at
 
están relacionados con pedidos cancelados, no entregados o en proceso, por lo que representan situaciones reales del negocio y no errores de calidad.

Dataset products

Existen 610 productos (1,85%) sin información descriptiva completa, pero su volumen es reducido y no justifica eliminar dichos registros.
 
# 4. La actividad comercial está altamente concentrada en pedidos entregados

El análisis de la variable order_status muestra que:
 
97,02% de los pedidos fueron entregados correctamente.
 
Los pedidos cancelados o no disponibles representan menos del 2% del total.

Esto indica una elevada eficiencia operativa de la plataforma y explica la presencia de algunos valores nulos en las fechas de entrega
 
# 5. Existen relaciones uno-a-muchos propias del negocio

Las tablas transaccionales presentan la estructura esperada en un ecommerce:

Orders → Items

112.650 líneas de pedido
98.666 pedidos únicos
 
Muchos pedidos contienen varios productos.
 
Orders → Payments
 
103.886 registros de pago
99.440 pedidos únicos
 
Algunos pedidos utilizan varios métodos de pago o pagos fraccionados.

Estas relaciones deberán tenerse en cuenta durante la fase de integración de datos para evitar duplicidades al realizar los merges.
 
# 6. Se han detectado pequeñas discrepancias que deberán revisarse

Durante la exploración se han identificado algunos casos que requieren validación en la fase de limpieza:

Aproximadamente 775 pedidos aparecen en orders pero no tienen registros asociados en items.
Existe un pedido sin información de pago.
Dos categorías de producto no encuentran correspondencia en la tabla de traducción.

Aunque su impacto es reducido, conviene analizarlos antes de construir el dataset final.
 
# 7. Los datos temporales son adecuados para el análisis

El periodo analizado comprende:
 
Septiembre de 2016 - Octubre de 2018
 
Este rango temporal permite estudiar:

evolución de ventas
estacionalidad
comportamiento mensual
impacto de los festivos

Además, el dataset de festivos cubre completamente dicho periodo, lo que permitirá enriquecer la información temporal.
 
# 8. El proyecto cumple sobradamente los requisitos mínimos

La combinación de:
 
Olist Ecommerce Dataset
+
Brazil Public Holidays Dataset
 
permite disponer de:

Más de 100.000 registros transaccionales.
Más de 20 variables analíticas.
Dos fuentes de datos independientes.
Posibilidad de realizar limpieza, análisis estadístico, visualización y construcción de dashboards.
